# 실습 3 · 최소제곱법으로 예측 모델 만들기
### Lab 3. Least Squares

**인공지능을 위한 선형대수학** · 9장 「최소제곱법과 정규방정식」 대응 실습

---

### 오늘의 목표

1. 정확한 해가 **없는** 문제를 직접 확인한다
2. **정규방정식**으로 최소제곱해를 구한다
3. 오차가 정말 **최소**인지, 정말 **직교**하는지 검증한다

> 💡 셀을 위에서부터 순서대로 **Shift + Enter**

## 0. 준비

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)

## 1. 사람을 한 명 더 추가하기

| 사람 | 몸무게 | 키 | 흡연 | 수명 |
|---|---|---|---|---|
| 1 | 60 | 5.5 | 1 | 66 |
| 2 | 65 | 5.0 | 0 | 74 |
| 3 | 55 | 6.0 | 1 | 78 |
| **4** | **50** | **5.0** | **1** | **72** |

이제 **방정식 4개, 미지수 3개** — 과결정 시스템입니다.

In [ ]:
A = np.array([[60, 5.5, 1],
              [65, 5.0, 0],
              [55, 6.0, 1],
              [50, 5.0, 1]])

b = np.array([66, 74, 78, 72])

print(A.shape, b.shape)   # (4, 3) (4,)

### 정말 해가 없을까?

아래 셀은 **일부러 오류를 내는** 셀입니다. `solve` 는 정사각행렬만 받습니다.

In [ ]:
print(np.linalg.solve(A, b))

> `LinAlgError: Last 2 dimensions of the array must be square`
>
> 4×3 은 정사각이 아니라 애초에 이 방법으로는 풀 수 없습니다.

## 2. 2장의 답을 그대로 써보면?

3명일 때 구했던 `x = [-0.4, 20, -20]` 을 4명 데이터에 적용해 봅니다.

In [ ]:
x_old = np.array([-0.4, 20, -20])

print("예측:", A @ x_old)
print("실제:", b)
print("오차:", b - A @ x_old)
print("오차의 길이:", np.linalg.norm(b - A @ x_old))

앞의 3명은 정확히 맞지만 **4번째 사람에서 12년이나 빗나갑니다.**

### 다른 답과 비교

In [ ]:
x_guess = np.array([-0.12, 16, -9.5])

print("오차:", b - A @ x_guess)
print("오차의 길이:", np.linalg.norm(b - A @ x_guess))

> 하나도 정확히 맞지 않지만 **오차의 길이는 더 작습니다.**
> 그럼 **가장 작게** 만드는 답은 무엇일까요?

## 3. 정규방정식 세우기

$$A^T A \hat{\mathbf{x}} = A^T \mathbf{b}$$

In [ ]:
C = A.T @ A     # 3 x 3 정사각행렬!
d = A.T @ b

print("A^T A =")
print(C)
print("크기:", C.shape)
print()
print("A^T b =", d)

> 🔑 **4×3 문제가 3×3 문제로 줄었습니다.**
> 데이터가 100만 개여도 `A^T A` 는 여전히 3×3 입니다.

In [ ]:
x_hat = np.linalg.solve(C, d)

print("최소제곱해 x_hat =", x_hat)
print("오차의 길이  =", np.linalg.norm(b - A @ x_hat))

### 세 답을 비교해 봅시다

In [ ]:
for name, x in [("2장의 답  ", x_old), ("찍어본 답 ", x_guess), ("최소제곱해", x_hat)]:
    print(name, "→ 오차의 길이 =", round(float(np.linalg.norm(b - A @ x)), 4))

> **최소제곱해가 가장 작습니다.** 그리고 이보다 더 작게 만드는 답은 **존재하지 않습니다.**

## 4. 정말 최소인지 확인해 보기

최소제곱해 주변을 조금씩 흔들어 보면 **반드시 오차가 커져야** 합니다.

In [ ]:
base = np.linalg.norm(b - A @ x_hat)
print("최소제곱해의 오차:", round(float(base), 6))
print()

for i in range(3):
    for step in (-0.5, 0.5):
        x_try = x_hat.copy()
        x_try[i] += step
        e = np.linalg.norm(b - A @ x_try)
        print(f"  x[{i}] {step:+.1f} → 오차 {e:.6f}   ({'더 큼 ✓' if e > base else '더 작음 ✗'})")

> 어느 쪽으로 흔들어도 **오차가 커집니다.** 정말 최소가 맞습니다.

## 5. 직교 조건 확인

수업에서 배운 핵심 — 오차 벡터는 `Col A` 에 **수직**이어야 합니다.

$$A^T(\mathbf{b} - A\hat{\mathbf{x}}) = \mathbf{0}$$

In [ ]:
r = b - A @ x_hat          # 오차(잔차) 벡터

print("오차 벡터 r =", r)
print("A^T r      =", A.T @ r)
print()
print("0에 가까운가?", np.allclose(A.T @ r, 0))

> ⚠️ 정확히 `0` 이 아니라 `1e-12` 같은 값이 나올 수 있습니다.
> 소수 계산의 미세한 오차이며, `np.allclose` 로 판단하는 이유입니다.

### 각 열과 하나씩 확인

In [ ]:
for j in range(A.shape[1]):
    print(f"{j}번 열 · 오차 =", round(float(A[:, j] @ r), 10))

## 6. 실무에서 쓰는 한 줄 — `lstsq`

정규방정식을 직접 세우지 않아도 됩니다.

In [ ]:
x_lstsq = np.linalg.lstsq(A, b, rcond=None)[0]

print("lstsq  :", x_lstsq)
print("정규방정식:", x_hat)
print("같은가?", np.allclose(x_lstsq, x_hat))

> 💡 실무에서는 **`lstsq` 를 씁니다.**
> `A^T A` 를 직접 만들면 계산 오차가 커질 수 있기 때문입니다.
>
> 하지만 **무슨 일이 일어나는지 아는 것**과 **버튼만 누르는 것**은 다릅니다.

## 7. 데이터가 많아지면

사람을 40명으로 늘려 봅시다. 진짜 규칙에 약간의 잡음을 섞어 만듭니다.

In [ ]:
rng = np.random.default_rng(0)          # 항상 같은 결과가 나오도록 고정

n = 40
weight  = rng.uniform(45, 90, n)
height  = rng.uniform(4.8, 6.2, n)
smoking = rng.integers(0, 2, n)

A20 = np.column_stack([weight, height, smoking])

true_x = np.array([-0.15, 16.5, -10.5])                 # 우리가 정한 '진짜 규칙'
b20 = A20 @ true_x + rng.normal(0, 1.0, n)              # 잡음을 섞음

print(A20.shape, b20.shape)

In [ ]:
x20 = np.linalg.lstsq(A20, b20, rcond=None)[0]

print("진짜 규칙 :", true_x)
print("찾아낸 답 :", x20)
print("오차의 길이:", round(float(np.linalg.norm(b20 - A20 @ x20)), 4))
print("A^T A 크기:", (A20.T @ A20).shape)

> 🎯 **잡음이 섞여 있는데도 진짜 규칙에 가까운 값을 찾아냈습니다.**
> 데이터가 40개인데도 `A^T A` 는 여전히 **3 × 3** 입니다.
>
> 이것이 **선형회귀(linear regression)** 이고, 머신러닝의 출발점입니다.

## 8. 직접 해보기 (과제)

**문제 1.** 수업 확인 문제를 코드로 풀고, 손으로 구한 답과 같은지 확인하시오.

$$A = \begin{bmatrix} 1 & 0 \\ 0 & 1 \\ 1 & 1 \end{bmatrix}, \qquad \mathbf{b} = \begin{bmatrix} 1 \\ 1 \\ 0 \end{bmatrix}$$

> `A.T @ A`, `A.T @ b`, `x_hat`, 오차의 길이를 모두 출력할 것 (손 계산 답: x_hat = [1/3, 1/3])

In [ ]:
# 여기에 코드를 쓰세요


**문제 2.** 문제 1의 오차 벡터가 `A` 의 두 열과 **직교**함을 코드로 확인하시오.

In [ ]:
# 여기에 코드를 쓰세요


**문제 3.** 7번의 40명 데이터에서 **잡음의 크기를 0으로** 바꾸면 (`rng.normal(0, 0.0, n)`)
찾아낸 답이 `true_x` 와 얼마나 가까워지는지 확인하고, 그 이유를 텍스트 셀에 쓰시오.

In [ ]:
# 여기에 코드를 쓰세요


---

### 오늘 배운 명령어

| 코드 | 뜻 |
|---|---|
| `A.T @ A`, `A.T @ b` | 정규방정식 세우기 |
| `solve(A.T@A, A.T@b)` | 정규방정식 풀기 |
| `np.linalg.lstsq(A, b, rcond=None)[0]` | **최소제곱해 한 줄로** (실무 표준) |
| `np.linalg.norm(b - A@x)` | 오차의 길이 |
| `np.column_stack([...])` | 열들을 옆으로 붙여 행렬 만들기 |
| `np.random.default_rng(0)` | 결과가 항상 같은 난수 생성기 |